In [ ]:
!pip install -U transformers datasets evaluate rouge-score nltk pandas matplotlib 

In [ ]:
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install -U git+https://github.com/huggingface/accelerate.git

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
import glob
from datasets import load_dataset
import datasets
from datasets import Dataset, concatenate_datasets
import evaluate

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [ ]:
!pip install -U evaluate

In [5]:
!git clone https://github.com/XueFuzhao/InstructionWild.git

Cloning into 'InstructionWild'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 259 (delta 3), reused 3 (delta 3), pack-reused 251
Receiving objects: 100% (259/259), 18.27 MiB | 16.93 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [6]:
from datasets import load_dataset

dataset = load_dataset('reddit')
print(f"Train dataset size: {len(dataset['train'])}")

Generating train split:   0%|          | 0/3848330 [00:00<?, ? examples/s]

Dataset reddit downloaded and prepared to /root/.cache/huggingface/datasets/reddit/default/1.0.0/bd1bf9097540c9101f329c123d12c6c6a042f65e5f0ab7f9bbabb0a54d3c840e. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

Train dataset size: 3848330


In [7]:
df1 = pd.read_json('InstructionWild/data_v2/user_4.jsonl', lines=True)
df2 = pd.read_json('InstructionWild/data_v2/user_1.jsonl', lines=True)

In [8]:
df2 = df2.drop('instruction_type', axis=1)
df2 = df2.drop('special_tag', axis=1)
df = pd.concat([df1, df2], axis=0)
df.columns = ['text']
df.reset_index(drop=True)
df['labels'] = '1'
dataset2 = Dataset.from_pandas(df)
dataset2 = dataset2.remove_columns(['__index_level_0__'])

In [9]:
dataset = dataset['train'].remove_columns(["body", "author", "normalizedBody", "subreddit", "subreddit_id", "id", "summary"])
dataset = dataset.rename_column("content", "text")
new_column = ['0'] * len(dataset)
dataset = dataset.add_column("labels", new_column)
dataset = dataset.select(range(len(dataset)//100))

In [10]:
dataset3 = concatenate_datasets([dataset, dataset2]).shuffle()

In [11]:
dataset3 = dataset3.train_test_split(test_size=0.1)

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id="google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [13]:

max_source_length = 512
print(f"Max source length: {max_source_length}")

max_target_length = 3
print(f"Max target length: {max_target_length}")

Max source length: 512
Max target length: 3


In [14]:
def preprocess_function(sample, padding="max_length"):
    inputs = [item for item in sample["text"]]

    model_inputs = tokenizer(inputs, max_length=max_source_length, padding=padding, truncation=True)

    labels = tokenizer(text_target=sample["labels"], max_length=max_target_length, padding=padding, truncation=True)
    
    if padding == "max_length":
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
        ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset3.map(preprocess_function, batched=True, remove_columns=['text', 'labels'])
print(f"Keys of tokenized dataset: {list(tokenized_dataset['train'].features)}")

Map:   0%|          | 0/134048 [00:00<?, ? examples/s]

Map:   0%|          | 0/14895 [00:00<?, ? examples/s]

Keys of tokenized dataset: ['labels', 'input_ids', 'attention_mask']


In [15]:
from transformers import AutoModelForSeq2SeqLM

model_id="google/flan-t5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

In [16]:
import evaluate
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize
nltk.download("punkt")

metric = evaluate.load("f1")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = ["\n".join(sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(sent_tokenize(label)) for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, average='macro')
    result = {k: round(v * 100, 4) for k, v in result.items()}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return result

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [17]:
from transformers import DataCollatorForSeq2Seq

label_pad_token_id = -100
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=label_pad_token_id,
    pad_to_multiple_of=8
)

In [18]:
from huggingface_hub import HfFolder
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

repository_id = f"{model_id.split('/')[1]}"

training_args = Seq2SeqTrainingArguments(
    output_dir=repository_id,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=False, 
    learning_rate=3e-4,

    num_train_epochs=1,
    logging_dir=f"{repository_id}/logs",
    logging_strategy="epoch", 
    evaluation_strategy="recall",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=repository_id,
    hub_token=HfFolder.get_token(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

Cloning https://huggingface.co/RottenLemons/flan-t5-base into local empty directory.


Download file pytorch_model.bin:   0%|          | 8.00k/945M [00:00<?, ?B/s]

Download file logs/events.out.tfevents.1689385206.92a4a412ef7c.210.2: 100%|##########| 463/463 [00:00<?, ?B/s]

Download file training_args.bin: 100%|##########| 4.06k/4.06k [00:00<?, ?B/s]

Download file logs/events.out.tfevents.1689374295.92a4a412ef7c.210.1: 100%|##########| 5.43k/5.43k [00:00<?, ?…

...

Clean file logs/events.out.tfevents.1689385206.92a4a412ef7c.210.2: 100%|##########| 463/463 [00:00<?, ?B/s]

Clean file training_args.bin:  25%|##4       | 1.00k/4.06k [00:00<?, ?B/s]

Download file spiece.model:   2%|2         | 16.5k/773k [00:00<?, ?B/s]

Clean file spiece.model:   0%|          | 1.00k/773k [00:00<?, ?B/s]

Clean file pytorch_model.bin:   0%|          | 1.00k/945M [00:00<?, ?B/s]

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
tokenizer.save_pretrained(repository_id)
trainer.create_model_card()
trainer.push_to_hub()

In [21]:
import torch
torch.cuda.empty_cache()

In [20]:
repository_id = f"{model_id.split('/')[1]}-upweighted"

training_args = Seq2SeqTrainingArguments(
    output_dir=repository_id,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=False, 
    learning_rate=3e-4,

    num_train_epochs=3,
    logging_dir=f"{repository_id}/logs",
    logging_strategy="epoch", 
    evaluation_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=repository_id,
    hub_token=HfFolder.get_token(),
)

trainer = UpWeightedSeq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

NameError: name 'UpWeightedSeq2SeqTrainer' is not defined

In [ ]:
trainer.train()

In [ ]:
# trainer.evaluate()

In [ ]:
# tokenizer.save_pretrained(repository_id)
# trainer.create_model_card()
# trainer.push_to_hub()

In [22]:
dataset4 = dataset.shuffle().select(range(len(dataset)//5))
dataset5 = concatenate_datasets([dataset4, dataset2]).shuffle()

In [23]:
dataset5 = dataset5.train_test_split(test_size=0.1)

In [25]:
# tokenized_inputs = concatenate_datasets([dataset5["train"], dataset5["test"]]).map(lambda x: tokenizer(x["text"], truncation=True), batched=True, remove_columns=['text', 'labels'])
max_source_length = 512
print(f"Max source length: {max_source_length}")

# tokenized_targets = concatenate_datasets([dataset5["train"], dataset5["test"]]).map(lambda x: tokenizer(x["labels"], truncation=True), batched=True, remove_columns=['text', 'labels'])
max_target_length = 3
print(f"Max target length: {max_target_length}")

Max source length: 512
Max target length: 3


In [26]:
tokenized_dataset = dataset5.map(preprocess_function, batched=True, remove_columns=['text', 'labels'])
print(f"Keys of tokenized dataset: {list(tokenized_dataset['train'].features)}")

Map:   0%|          | 0/106340 [00:00<?, ? examples/s]

Map:   0%|          | 0/11816 [00:00<?, ? examples/s]

Keys of tokenized dataset: ['labels', 'input_ids', 'attention_mask']


In [ ]:
# from transformers import DataCollatorForSeq2Seq

# label_pad_token_id = -100
# data_collator = DataCollatorForSeq2Seq(
#     tokenizer,
#     model=model,
#     label_pad_token_id=label_pad_token_id,
#     pad_to_multiple_of=8
# )

In [ ]:
# class DownSampledSeq2SeqTrainer(Seq2SeqTrainer):
#     def __init__(self, config=None, data_args=None, *args, **kwargs):
#         super().__init__(*args, **kwargs)

#         if config is None:
#             assert isinstance(self.model, PreTrainedModel), (
#                 "If no `config` is passed the model to be trained has to be of type `PreTrainedModel`, but is"
#                 f" {self.model.__class__}"
#             )
#             self.config = self.model.config
#         else:
#             self.config = config

#         self.data_args = data_args
#         self.vocab_size = self.config.tgt_vocab_size if isinstance(self.config, FSMTConfig) else self.config.vocab_size

#         if self.args.label_smoothing != 0 or (self.data_args is not None and self.data_args.ignore_pad_token_for_loss):
#             assert self.config.pad_token_id is not None, (
#                 "Make sure that `config.pad_token_id` is correcly defined when ignoring `pad_token` for loss"
#                 " calculation or doing label smoothing."
#             )

#         if self.config.pad_token_id is None and self.config.eos_token_id is not None:
#             logger.warning(
#                 f"The `config.pad_token_id` is `None`. Using `config.eos_token_id` = {self.config.eos_token_id} for"
#                 " padding.."
#             )

#         if self.args.label_smoothing == 0:
#             self.loss_fn = torch.nn.CrossEntropyLoss(ignore_index=self.config.pad_token_id, weight=torch.tensor([1.0, 7.0]))
#         else:
#             # dynamically import label_smoothed_nll_loss
#             from utils import label_smoothed_nll_loss

#             self.loss_fn = label_smoothed_nll_loss


In [ ]:
# from huggingface_hub import HfFolder
# from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# repository_id = f"{model_id.split('/')[1]}-mahir-downsample"

# training_args = Seq2SeqTrainingArguments(
#     output_dir=repository_id,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     predict_with_generate=True,
#     fp16=False, 
#     learning_rate=3e-4,

#     num_train_epochs=3,
#     logging_dir=f"{repository_id}/logs",
#     logging_strategy="epoch", 
#     evaluation_strategy="no",
#     save_strategy="epoch",
#     save_total_limit=2,
#     load_best_model_at_end=False,
#     push_to_hub=True,
#     hub_strategy="every_save",
#     hub_model_id=repository_id,
#     hub_token=HfFolder.get_token(),
# )

# trainer = DownSampledSeq2SeqTrainer(
#     model=model,
#     args=training_args,
#     data_collator=data_collator,
#     train_dataset=tokenized_dataset["train"],
#     eval_dataset=tokenized_dataset["test"],
#     compute_metrics=compute_metrics,
# )

In [ ]:
# trainer.train()

In [ ]:
# trainer.evaluate()

In [ ]:
# tokenizer.save_pretrained(repository_id)
# trainer.create_model_card()
# trainer.push_to_hub()

In [27]:
from huggingface_hub import HfFolder
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

repository_id = f"{model_id.split('/')[1]}-downsamples"

training_args = Seq2SeqTrainingArguments(
    output_dir=repository_id,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=False, 
    learning_rate=3e-4,

    num_train_epochs=1,
    logging_dir=f"{repository_id}/logs",
    logging_strategy="epoch", 
    evaluation_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=repository_id,
    hub_token=HfFolder.get_token(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

Cloning https://huggingface.co/RottenLemons/flan-t5-base-downsamples into local empty directory.


Download file pytorch_model.bin:   0%|          | 1.40k/945M [00:00<?, ?B/s]

Download file logs/events.out.tfevents.1689391146.92a4a412ef7c.210.9: 100%|##########| 5.49k/5.49k [00:00<?, ?…

Download file training_args.bin: 100%|##########| 4.06k/4.06k [00:00<?, ?B/s]

Download file logs/events.out.tfevents.1689400132.92a4a412ef7c.210.10: 100%|##########| 456/456 [00:00<?, ?B/s…

...

Clean file logs/events.out.tfevents.1689400132.92a4a412ef7c.210.10: 100%|##########| 456/456 [00:00<?, ?B/s]

Clean file logs/events.out.tfevents.1689391146.92a4a412ef7c.210.9:  18%|#8        | 1.00k/5.49k [00:00<?, ?B/s…

Clean file training_args.bin:  25%|##4       | 1.00k/4.06k [00:00<?, ?B/s]

Clean file spiece.model:   0%|          | 1.00k/773k [00:00<?, ?B/s]

Clean file pytorch_model.bin:   0%|          | 1.00k/945M [00:00<?, ?B/s]

In [28]:
trainer.train()

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
13293,0.017400


TrainOutput(global_step=13293, training_loss=0.017424902330893728, metrics={'train_runtime': 8855.6493, 'train_samples_per_second': 12.008, 'train_steps_per_second': 1.501, 'total_flos': 7.281709685932032e+16, 'train_loss': 0.017424902330893728, 'epoch': 1.0})

In [29]:
trainer.evaluate()

{'eval_loss': 0.004144715610891581,
 'eval_f1': 99.1107,
 'eval_gen_len': 2.062965470548409,
 'eval_runtime': 617.0798,
 'eval_samples_per_second': 19.148,
 'eval_steps_per_second': 2.394,
 'epoch': 1.0}

In [30]:
tokenizer.save_pretrained(repository_id)
trainer.create_model_card()
trainer.push_to_hub()

Upload file logs/events.out.tfevents.1689421062.72a9468c8cc3.154.0: 100%|##########| 5.47k/5.47k [00:00<?, ?B/…

Upload file logs/events.out.tfevents.1689430534.72a9468c8cc3.154.1: 100%|##########| 456/456 [00:00<?, ?B/s]

To https://huggingface.co/RottenLemons/flan-t5-base-downsamples
   e001fa9..1eab944  main -> main



'https://huggingface.co/RottenLemons/flan-t5-base-downsamples/commit/1eab94409c89f30be296f9e5d95fdc8fe0741ca4'